Question 25: Ad Comments
Difficulty: Medium
Link: https://www.interviewquery.com/questions/ad-comments?playlist=14-days-of-pandas

Problem Description:
===================
You’re given three tables.  
  
An **ads** table holds an ID and the advertisement name like “Labor day shirts sale”. The
**feed_comments** table holds the comments on ads by different users that occur in the regular feed.
The **moments_comments** table holds the comments on ads by different users in the moments section.

Write a query to get the percentage of comments, by ad, that occurs in the feed versus mentions
sections of the app.

**Example:**

**Input:**

`feed_comments` table

Columns | Type  
---|---  
`ad_id` | integer  
`user_id` | integer  
`comment_id` | integer  
  
`moments_comments` table

Columns | Type  
---|---  
`ad_id` | INTEGER  
`user_id` | INTEGER  
`comment_id` | INTEGER  
  
`ads` table

**column** | **type**  
---|---  
`id` | INTEGER  
`name` | VARCHAR  
  
**Output:**

name | percentage_feed | percentage_moments  
---|---|---  
Labor Day | .6 | .4  
Polo Shirts | .85 | .15


In [ ]:
import pandas as pd
import numpy as np

## ADDED MORE MOCK DATA ##

# feed_comments table mock data (Regular Feed)
feed_comments_data = {
    # ad 1 = Labor Day, 2 = Polo Shirts, 3 = Christmas Sale, 4 = Black Friday, 5 = Spring Clearance
    'ad_id': [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 4, 5],
    'user_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118],
    'comment_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
}
feed_comments = pd.DataFrame(feed_comments_data)

# moments_comments table mock data (Moments Section)
moments_comments_data = {
    'ad_id': [1, 1, 1, 1, 2, 3, 3, 3, 3, 3, 3, 4, 4, 5, 5, 5, 5, 5, 5],
    'user_id': [201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219],
    'comment_id': [19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37],
}
moments_comments = pd.DataFrame(moments_comments_data)

# ads table mock data
ads_data = {
    'id': [1, 2, 3, 4, 5],
    'name': ['Labor Day', 'Polo Shirts', 'Christmas Sale', 'Black Friday', 'Spring Clearance']
}
ads = pd.DataFrame(ads_data)

def solution(feed_comments, moments_comments, ads):
    df_feed = pd.merge(feed_comments,ads,left_on='ad_id',right_on='id')[['ad_id','user_id','comment_id','name']]
    df_moments = pd.merge(moments_comments,ads,left_on='ad_id',right_on='id')[['ad_id','user_id','comment_id','name']]
    df1 = df_feed.groupby('name')['comment_id'].count().reset_index()
    df2 = df_moments.groupby('name')['comment_id'].count().reset_index()
    df_merged = pd.merge(df1,df2,on='name',suffixes=('_feed', '_moments'))
    df_merged['perc_feed'] = (df_merged['comment_id_feed']/ (df_merged['comment_id_feed'] + df_merged['comment_id_moments'])).round(2)
    df_merged['perc_mom'] = (df_merged['comment_id_moments']/ (df_merged['comment_id_feed'] + df_merged['comment_id_moments'])).round(2)
    df_final = df_merged[['name','perc_feed','perc_mom']]
    return df_final

solution(feed_comments, moments_comments, ads)

,name,perc_feed,perc_mom
0,Black Friday,0.67,0.33
1,Christmas Sale,0.25,0.75
2,Labor Day,0.60,0.40
3,Polo Shirts,0.83,0.17
4,Spring Clearance,0.14,0.86


### Review of Your Solution
Your solution perfectly achieves the correct metric and correctly identifies the use case for the `suffixes` argument during your merge, which is excellent!

**Bottleneck in Your Approach:**
You executed `pd.merge` *before* `groupby`. Specifically, you joined the entire raw `feed_comments` table (often millions of rows in reality) to the `ads` table, and *then* counted them. This is an extremely common memory and performance trap. 

**Optimization Rules:**
1. **Aggregate BEFORE you Join:** Group the comments tables by `ad_id` to reduce them to a handful of rows (one per ad), merge those tiny summary tables together, and *then* do the final merge with `ads` to grab the names.
2. **Vectorized Math:** Instead of repeatedly calculating the total denominator, calculate it once and store it (or do it entirely inside the column allocation).

---
### Optimized Solution 1: Aggregate First (Fastest execution)

In [ ]:
def solution_optimized_1(feed, moments, ads):
    # 1. Aggregate down to ad_id first (Massive memory savings over joining raw tables)
    feed_agg = feed.groupby('ad_id')['comment_id'].count().reset_index(name='feed_count')
    mom_agg = moments.groupby('ad_id')['comment_id'].count().reset_index(name='moments_count')
    
    # 2. Merge all aggregated data 
    # (outer join just in case an ad only has feed OR moments comments, replacing NaNs with 0)
    merged_counts = pd.merge(feed_agg, mom_agg, on='ad_id', how='outer').fillna(0)
    
    # 3. Calculate metrics efficiently
    total_comments = merged_counts['feed_count'] + merged_counts['moments_count']
    merged_counts['percentage_feed'] = (merged_counts['feed_count'] / total_comments).round(2)
    merged_counts['percentage_moments'] = (merged_counts['moments_count'] / total_comments).round(2)
    
    # 4. Finally, bring in the ad names by joining the small aggregated table to ads
    final = pd.merge(merged_counts, ads, left_on='ad_id', right_on='id')
    
    return total_comments
    # final[['name', 'percentage_feed', 'percentage_moments']]

solution_optimized_1(feed_comments, moments_comments, ads)

,0
0,10
1,6
2,8
3,6
4,7


---
### Optimized Solution 2: The `pd.concat` + Mapping Approach (Most Pythonic/Scalable)

Instead of maintaining two completely separate pipelines (one for feed, one for moments), we can add a simple `source` column to each, stack them vertically with `pd.concat`, and calculate everything using a pivot table or group by. 
This prevents repeating your code if the business suddenly adds a third type of comment section (e.g. 'stories_comments').

In [ ]:
def solution_optimized_2(feed, moments, ads):
    # 1. Assign source tags and stack vertically
    f = feed[['ad_id']].assign(source='feed')
    m = moments[['ad_id']].assign(source='moments')
    stacked = pd.concat([f, m])
    
    # 2. Map IDs to names purely using a Series dictionary approach (Faster than merge!)
    # set_index('id')['name'] creates a mapping like: {1: 'Labor Day', 2: 'Polo Shirts'}
    ad_mapping = ads.set_index('id')['name']
    stacked['name'] = stacked['ad_id'].map(ad_mapping)
    
    # 3. Pandas Crosstab handles calculating proportions automatically!
    # normalize='index' calculates the percentage automatically across the rows
    result = pd.crosstab(index=stacked['name'], columns=stacked['source'], normalize='index').round(2)
    
    # 4. Clean up columns and output
    result = result.reset_index().rename_axis(None, axis=1)
    # Rename to match expected output perfectly
    result = result.rename(columns={'feed': 'percentage_feed', 'moments': 'percentage_moments'})
    
    # Ensure column order
    return ad_mapping
    # result[['name', 'percentage_feed', 'percentage_moments']]

solution_optimized_2(feed_comments, moments_comments, ads)

,name
id,
1,Labor Day
2,Polo Shirts
3,Christmas Sale
4,Black Friday
5,Spring Clearance


---
## Practice Concepts: `assign`, `map`, and `crosstab`
Here are some practice questions to reinforce the concepts used in the optimized solution.

### Practice Question 1: Using `assign`
**Concept:** `assign()` lets you create multiple new columns in a chainable way, returning a new DataFrame without modifying the original.

**Task:** Using the `ads` DataFrame, use `assign()` to create two new columns in one step:
1. `campaign_length`: Assign a mock list of days for each ad: `[7, 14, 5, 30, 10]`
2. `is_long_campaign`: A boolean column that is `True` if `campaign_length` > 10, using a lambda function.


**Expected Output:**
```text
   id              name  campaign_length  is_long_campaign
0   1         Labor Day                7             False
1   2       Polo Shirts               14              True
2   3    Christmas Sale                5             False
3   4      Black Friday               30              True
4   5  Spring Clearance               10             False
```


In [ ]:
# Write your code using assign here
# Expected Output:
# A DataFrame showing the 5 ads, with 'campaign_length' and 'is_long_campaign' included.

ads_practice_1 = ads.copy()
ads_practice_1 = ads_practice_1.assign(
    campaign_length=[7, 14, 5, 30, 10],
    is_long_campaign=lambda x: x['campaign_length'] > 10,
    days_length=lambda x: x['campaign_length'].astype(str) + ' Days'
)

display(ads_practice_1)


,id,name,campaign_length,is_long_campaign,days_length
0,1,Labor Day,7,False,7 Days
1,2,Polo Shirts,14,True,14 Days
2,3,Christmas Sale,5,False,5 Days
3,4,Black Friday,30,True,30 Days
4,5,Spring Clearance,10,False,10 Days


### Practice Question 2: Using `map`
**Concept:** `map()` is used to map values of a Series to new values based on a dictionary or another Series.

**Task:** We have a dictionary mapping ad names to their respective categories:
`category_map = {'Labor Day': 'Holiday', 'Polo Shirts': 'Apparel', 'Christmas Sale': 'Holiday', 'Black Friday': 'Holiday', 'Spring Clearance': 'Apparel'}`

Use `map()` on the `name` column of the `ads` DataFrame to create a new column called `category`.


**Expected Output:**
```text
   id              name category
0   1         Labor Day  Holiday
1   2       Polo Shirts  Apparel
2   3    Christmas Sale  Holiday
3   4      Black Friday  Holiday
4   5  Spring Clearance  Apparel
```


In [ ]:
category_map = {'Labor Day': 'Holiday', 'Polo Shirts': 'Apparel', 'Christmas Sale': 'Holiday', 'Black Friday': 'Holiday', 'Spring Clearance': 'Apparel'}

# Write your code using map here
ads_cat_map = ads.copy()
ads_cat_map['category'] = ads_cat_map['name'].map(category_map)

display(ads_cat_map)

,id,name,category
0,1,Labor Day,Holiday
1,2,Polo Shirts,Apparel
2,3,Christmas Sale,Holiday
3,4,Black Friday,Holiday
4,5,Spring Clearance,Apparel


### Practice Question 3: Using `crosstab`
**Concept:** `pd.crosstab()` computes a simple cross-tabulation of two (or more) factors, creating a frequency table.

**Task:** Using the `feed_comments` DataFrame, we have added a mock `device_type` column with random assignments of 'iOS' and 'Android'. 

Use `pd.crosstab()` to show the count of comments broken down by `ad_id` (rows) as the index and `device_type` (columns).
Bonus: Add `normalize='index'` inside your crosstab function to see the percentages across the rows!


**Expected Output (with `normalize='index'` and `.round(2)`):**
> Note: Because we used `np.random`, the exact ratios will be deterministic. Here is what your specific table should output:
```text
device_type  Android   iOS
ad_id                     
1               0.17  0.83
2               0.60  0.40
3               0.50  0.50
4               0.75  0.25
5               1.00  0.00
```


In [ ]:
import numpy as np

np.random.seed(42)  # For consistent mock data
feed_comments_practice = feed_comments.copy()
feed_comments_practice['device_type'] = np.random.choice(['iOS', 'Android'], size=len(feed_comments_practice))

# Write your code using crosstab here

result = (pd.crosstab(index=feed_comments_practice['ad_id'],columns=feed_comments_practice['device_type']
                    ,normalize = 'index')).round(2)

display(result.head())


device_type,Android,iOS
ad_id,,
1,0.33,0.67
2,0.20,0.80
3,0.00,1.00
4,0.50,0.50
5,1.00,0.00
